In [1]:
%load_ext mam_graph

In [2]:
from datetime import datetime
from pathlib import Path

from supermam.connectivity import process_synapse_numbers
from supermam.cytoarchitecture import process_neuron_numbers
from supermam.experimental_data_packet.humam.params import params as humam_params
from supermam.network import load_network, process_network, scale_network

from mam_graph import EdgeBundle
from converter import mam_to_digraph

In [3]:
force_processing = False
model = "humam"

# Define output path for data and parameters.
# outpath = Path("out") / f"run_downscaled_{model}/{tic.strftime('%y%m%d-%H%M%S')}"
outpath = Path("out") / f"run_downscaled_{model}"
humam_params.network_params.outpath = outpath / "network"

if not outpath.exists():
    outpath.mkdir(parents=True, exist_ok=True)

if not force_processing and humam_params.network_params.outpath.exists():
    # Load network
    network = load_network(humam_params.network_params.outpath)
else:
    # Calculates the number of neurons in the network
    NN = process_neuron_numbers(humam_params.neuron_numbers_params)

    # Calculates the number of synapses in the network
    SN = process_synapse_numbers(NN, humam_params.synapse_numbers_params)

    # Process fullscaled network
    fullscaled_network = process_network(NN, SN, humam_params.network_params)

    # # Scale network
    network = scale_network(fullscaled_network, 0.004)

    # # Dump data packet to files
    network.model_dump_packet()

In [4]:
graph = mam_to_digraph(network)

In [5]:
%edgebundle graph --levels area --beta 0.85 --size 1200

<EdgeBundle: 136 leaves, 4048 links>